# Reshaping and Wrangling

Data can be organised in either wide or long format. The most useful format depends on how the data were collected and what you want to do with them.

### Wide format

The `messyDat` dataset is in wide format. Each row represents one participant, and each type of information is stored in a separate column.

The reaction times are stored in two different columns:

- `congruent_rt`: reaction time when the font colour matches the word.
- `incongruent_rt`: reaction time when the font colour and word do not match.

| pcp_id | pcp_age | congruent_rt_ | incongruent_rt | pcp_edu_level | exp_completed |
|---|---|---:|---:|---|---|
| P001 | 18 | 2.45 | 3.25 | UG | TRUE |
| P002 | 19 | 1.89 | 2.99| PG | TRUE |

Wide format is convenient for data entry because all information from one participant appears on one row. However, the Stroop condition is contained in the column names rather than in a separate variable.

### Long format

In long format, each participant has one row for each observation. Rather than storing reaction times in two separate columns, we use:

- one column to identify the condition;
- one column to store the reaction time.

| pcp_id | pcp_age | pcp_edu_level | exp_completed | condition | reaction_time |
|---|---:|---|---|---|---:|
| P001 | 18 | UG | TRUE | congruent | 2.45 |
| P001 | 18 | UG | TRUE | incongruent | 3.25 |
| P002 | 19 | PG | TRUE | congruent | 1.89 |
| P002 | 19 | PG | TRUE | incongruent | 2.99 |

Each participant now appears twice: once for their congruent reaction time and once for their incongruent reaction time.

### Why use long format in R?

Long format is usually more useful for analysis and plotting because each column represents one variable:

- `pcp_id` identifies the participant.
- `condition` identifies whether the trial was congruent or incongruent.
- `reaction_time` contains the measured response.

This makes it easier to group, summarise, plot, and model the data:

The key difference is:

- Wide format: one row per participant, with separate columns for each repeated measurement.
- Long format: multiple rows per participant, with one column for the repeated measurement and another column describing its condition.


## Reshaping data from wide to long

More often than not if you want to switch between formats you will be going from wide to long. The `reshape()` function can do both directions but we'll concentrate on wide to long in this section.

| Argument | Purpose in wide-to-long reshaping | Example |
|---|---|---|
| `data` | The data frame to reshape. | `data = messyDat` |
| `varying` | Columns containing repeated measurements that will become one column. | `varying = c("congruent_rt", "incongruent_rt")` |
| `v.names` | Name of the new column containing values from the repeated-measurement columns. | `v.names = "reaction_time"` |
| `timevar` | Name of the new column that identifies the condition. | `timevar = "condition"` |
| `times` | Labels to place in `timevar`, in the same order as `varying`. | `times = c("congruent", "incongruent")` |
| `idvar` | Column that identifies each participant or original row. | `idvar = "pcp_id"` |
| `direction` | Direction of conversion. Use `"long"` for wide-to-long. | `direction = "long"` |


```r
messyDatLong <- reshape(
  data = messyDat,
  varying = c("congruent_rt", "incongruent_rt"),
  v.names = "reaction_time",
  timevar = "condition",
  times = c("congruent", "incongruent"),
  idvar = "pcp_id",
  direction = "long"
)

rownames(messyDatLong) <- NULL 

```
The code above creates a new dataframe called `messyDatLong` which is now in the long format. The `rownames(messyDatLong) <- NULL ` command removes row names from the data frame that `reshape()` adds to help you see what it has done. In this case we don't really need that information so we have removed them. 

We can verify that the data frame is now in long format with `str()`

```r
str(messyDatLong)
```

```text
'data.frame':	14 obs. of  6 variables:
 $ pcp_id       : chr  "P001" "P002" "P003" "P004" ...
 $ pcp_age      : num  18 19 NA 20 22 20 18 18 19 NA ...
 $ pcp_edu_level: Ord.factor w/ 3 levels "School"<"UG"<..: 1 2 3 3 2 1 1 1 2 3 ...
 $ completed_exp: logi  TRUE TRUE TRUE TRUE TRUE TRUE ...
 $ condition    : chr  "congruent" "congruent" "congruent" "congruent" ...
 $ reaction_time: num  2.45 1.89 2.1 3 2.1 1.77 2.13 3.25 2.99 NA ...
 - attr(*, "reshapeLong")=List of 4
  ..$ varying:List of 1
  .. ..$ reaction_time: chr [1:2] "congruent_rt" "incongruent_rt"
  .. ..- attr(*, "v.names")= chr "reaction_time"
  .. ..- attr(*, "times")= chr [1:2] "congruent" "incongruent"
  ..$ v.names: chr "reaction_time"
  ..$ idvar  : chr "pcp_id"
  ..$ timevar: chr "condition"
```

The output tells us that there are now twice as many rows (2 per participant) and we have new columns `condition` and `reaction_time`. One to identify the experimental condition and one to identify the associated reaction time. How the reshape was performed has also been stored as metadata.

There is one remaining issue which you may have noticed. `condition` is chr but it should really be a factor. Let's fix that:

```r
messyDatLong$condition <- factor(messyDatLong$condition)
levels(messyDatLong$condition)
```
```text
[1] "congruent"   "incongruent"
```
congruent and incongruent are not naturally ordered so we can leave the default alphabetical ordering used by the `order()` function.

## Creating new variables

Sometimes it is helpful to add new variables to encode something about your data. Imagine we read that Stroop trials with a reaction time over 3 seconds should not be included because it means participants are not following the instruction to respond as fast as possible. We can create a new column to encode whether this is true or not for each row.

```r
messyDatLong$slow <- messyDatLong$reaction_time > 3
```
The right hand side tests whether values in the reaction_time column are over 3 seconds. The resulting vector of `TRUE` and `FALSE` values is then assigned to `messyDatLong$slow`. There was no `slow` column in `messyDatLong` so R simply created one. Had there already been a column called `slow` it would have been overwritten.

We can verify that this has worked using the `head()` function which simply displays the first few rows of the data frame:

```r
head(messyDatLong)
```

```text
  pcp_id pcp_age pcp_edu_level completed_exp condition reaction_time  slow
1   P001      18        School          TRUE congruent          2.45 FALSE
2   P002      19            UG          TRUE congruent          1.89 FALSE
3   P003      NA            PG          TRUE congruent          2.10 FALSE
4   P004      20            PG          TRUE congruent          3.00 FALSE
5   P005      22            UG          TRUE congruent          2.10 FALSE
6   P006      20        School          TRUE congruent          1.77 FALSE
```

### Creating separate variables from one column

Sometimes a column will contain information about more than one variable. The example generated below contains accuracy data from two participants who completed four experimental conditions. The `condition` column combines two pieces of information:

- Cue type: `Visual` or `Audio`
- Difficulty: `Easy` or `Hard`

```r
exp <- data.frame(
  participant_id = rep(c(1, 2), each = 4),
  condition = c(
    "VisualEasy",
    "VisualHard",
    "AudioEasy",
    "AudioHard",
    "VisualEasy",
    "VisualHard",
    "AudioEasy",
    "AudioHard"
  ),
  accuracy = c(92, 75, 88, 70, 95, 68, 78, 90)
)

head(exp)
```
```text
  participant_id  condition accuracy
1              1 VisualEasy       92
2              1 VisualHard       75
3              1  AudioEasy       88
4              1  AudioHard       70
5              2 VisualEasy       95
6              2 VisualHard       68
```

As it stands the `condition` column contains information about two independent variables: Cue Type and Difficulty. We always want our independent variables to have their own column/variable in R data frames.

The code below creates two new variables, `cue_type` and `difficulty`, from the `condition` column.

```r

exp$cue_type[exp$condition %in% c("VisualEasy", "VisualHard")] <- "visual"
exp$cue_type[exp$condition %in% c("AudioEasy", "AudioHard")] <- "audio"
exp$difficulty[exp$condition %in% c("VisualEasy", "AudioEasy")] <- "easy"
exp$difficulty[exp$condition %in% c("VisualHard", "AudioHard")] <- "hard"
```



The `%in%` operator checks whether each value appears in a specified list of values.

```r
exp$condition %in% c("VisualEasy", "VisualHard")
```

This produces `TRUE` for rows where `condition` is `"VisualEasy"` or `"VisualHard"`, and `FALSE` for all other rows.

```r
exp$cue_type[exp$condition %in% c("VisualEasy", "VisualHard")] <- "visual"
```

This line has three parts:

1. `exp$cue_type` refers to a column called `cue_type` in the data frame `exp`. If it does not yet exist, R creates it.

2. The square brackets `[ ]` select only the rows where the condition inside them is `TRUE`.

3. `<- "visual"` assigns `"visual"` to the selected rows.

Let's check it produced the desired effect:

```r
head(exp)
```
```text
  participant_id  condition accuracy cue_type difficulty
1              1 VisualEasy       92   visual       easy
2              1 VisualHard       75   visual       hard
3              1  AudioEasy       88    audio       easy
4              1  AudioHard       70    audio       hard
5              2 VisualEasy       95   visual       easy
6              2 VisualHard       68   visual       hard
```